# SignSpeak - CNN training on the ASL Alphabet dataset

Trains a convolutional neural network on the [Kaggle ASL Alphabet dataset](https://www.kaggle.com/datasets/grassknoted/asl-alphabet) (87k images, 29 classes: A-Z plus space, delete, nothing).

This is the image-based model from the original SignSpeak prototype. The live demo in this repo uses the landmark classifier by default (trains in seconds on your own samples), but this CNN is stronger if you want to recognize signs from full images instead of MediaPipe landmarks.

Run in Google Colab with a GPU runtime.

## 1. Get the dataset

Download `archive.zip` from the Kaggle dataset page (free account required) and upload it to your Google Drive, then mount Drive here. Update `DATA_DIR` below to where you extracted `asl_alphabet_train`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/asl_alphabet_train"  # update to your path
IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 15  # use a small number (e.g. 3) for a quick demo run

## 2. Build the data pipeline

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
)

train_gen = datagen.flow_from_directory(
    DATA_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    color_mode="rgb", class_mode="categorical", subset="training")

val_gen = datagen.flow_from_directory(
    DATA_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    color_mode="rgb", class_mode="categorical", subset="validation")

## 3. Define and train the CNN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(29, activation="softmax"),  # A-Z + space / delete / nothing
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(train_gen, epochs=EPOCHS, validation_data=val_gen)

## 4. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.title("Accuracy"); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.title("Loss"); plt.legend()
plt.show()

## 5. Save the model

In [ ]:
model.save("asl_cnn_model.h5")
print("saved asl_cnn_model.h5")

## 6. Try it on a frame

Crop a hand region (the demo app uses MediaPipe to find the bounding box), resize to 64x64, and predict:

In [ ]:
import numpy as np
import cv2
from tensorflow.keras.models import load_model

labels = [chr(i) for i in range(ord("A"), ord("Z") + 1)] + ["space", "delete", "nothing"]
model = load_model("asl_cnn_model.h5")

def predict_letter(hand_img_bgr):
    img = cv2.resize(hand_img_bgr, (IMG_SIZE, IMG_SIZE)) / 255.0
    probs = model.predict(np.expand_dims(img, axis=0))[0]
    return labels[int(np.argmax(probs))], float(np.max(probs))